In [1]:
import numpy as np
import pandas as pd
from sgp4.api import Satrec
from astropy.time import Time
from astropy.coordinates import TEME, ITRS, EarthLocation, get_sun
import astropy.units as u

# ==========================================
# 解决卡顿的核心代码：强制关闭 IERS 联网下载
from astropy.utils.iers import conf
conf.auto_download = False 
# ==========================================

# --- 1. 输入典型太阳同步轨道 TLE ---
s = '1 42017U 17008C   26114.50000000  .00000000  00000-0  50000-4 0  9998'
t = '2 42017  97.5000 100.0000 0010000   0.0000 180.0000 15.20000000500003'
satellite = Satrec.twoline2rv(s, t)

# --- 2. 设定仿真的时间步长 ---
start_time = Time('2026-04-24T10:00:00', scale='utc')
time_steps = start_time + np.arange(0, 100) * u.minute

results = []

print("正在计算轨道和太阳几何，请稍候...")

for t_current in time_steps:
    # --- 3. SGP4 轨道预报 ---
    jd1, jd2 = t_current.jd1, t_current.jd2
    e, r_teme, v_teme = satellite.sgp4(jd1, jd2)
    
    if e != 0:
        continue 
        
    # --- 4. 坐标系转换 (TEME -> ITRS) ---
    teme_coord = TEME(x=r_teme[0]*u.km, y=r_teme[1]*u.km, z=r_teme[2]*u.km, obstime=t_current)
    itrs_coord = teme_coord.transform_to(ITRS(obstime=t_current))
    
    earth_loc = itrs_coord.earth_location
    lon = earth_loc.lon.value
    lat = earth_loc.lat.value
    alt = earth_loc.height.to_value(u.km)
    
    # --- 5. 计算太阳几何角基础 ---
    sun_pos = get_sun(t_current)
    sun_itrs = sun_pos.transform_to(ITRS(obstime=t_current))
    
    results.append({
        'UTC Time': t_current.isot,
        'Sat_Lon (deg)': round(lon, 3),
        'Sat_Lat (deg)': round(lat, 3),
        'Sat_Alt (km)': round(alt, 3),
        'Sun_X (km)': round(sun_itrs.x.value, 1),
        'Sun_Y (km)': round(sun_itrs.y.value, 1),
        'Sun_Z (km)': round(sun_itrs.z.value, 1)
    })

# --- 6. 整理输出 ---
df_orbit = pd.DataFrame(results)
print("计算完成！")
display(df_orbit.head(10))

正在计算轨道和太阳几何，请稍候...
计算完成！


,UTC Time,Sat_Lon (deg),Sat_Lat (deg),Sat_Alt (km),Sun_X (km),Sun_Y (km),Sun_Z (km)
0,2026-04-24T10:00:00.000,44.392,80.699,518.546,0.9,0.5,0.2
1,2026-04-24T10:01:00.000,20.750,82.330,519.122,0.9,0.5,0.2
2,2026-04-24T10:02:00.000,-8.032,82.281,519.575,0.9,0.5,0.2
3,2026-04-24T10:03:00.000,-31.196,80.577,519.904,0.9,0.5,0.2
4,2026-04-24T10:04:00.000,-45.893,77.899,520.109,0.9,0.5,0.2
5,2026-04-24T10:05:00.000,-55.078,74.752,520.193,0.9,0.5,0.2
6,2026-04-24T10:06:00.000,-61.179,71.370,520.162,0.9,0.5,0.2
7,2026-04-24T10:07:00.000,-65.510,67.861,520.025,0.9,0.5,0.2
8,2026-04-24T10:08:00.000,-68.764,64.276,519.791,0.9,0.5,0.2
9,2026-04-24T10:09:00.000,-71.323,60.642,519.472,0.9,0.4,0.2
